# 推荐模型还原实战：饿了么 DIN vs 美团 MMOE

本 Notebook 将指导你如何利用天池电商数据集（`Rec-Tmall`），还原中国两大外卖平台的典型推荐模型架构。

1.  **数据准备**：构造用户行为序列（用于 DIN）和多目标标签（用于 MMOE）。
2.  **饿了么风格 (DIN)**：实现 Deep Interest Network，捕捉用户对当前商品的动态兴趣。
3.  **美团风格 (MMOE)**：实现 Multi-gate Mixture-of-Experts，同时优化点击率 (CTR) 和转化率 (CVR)。

In [ ]:
%pip install pandas numpy tensorflow scikit-learn
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. 加载数据
data_path = r'DATASET\Rec-Tmall\(sample)sam_tianchi_2014002_rec_tmall_log.csv'
df = pd.read_csv(data_path)

# 简单预处理：填充缺失值，编码 ID
for col in ['user_id', 'item_id', 'action']:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

print("数据概览:")
print(df.head())
print(f"总行数: {len(df)}")

# 定义常量
NUM_USERS = df['user_id'].max() + 1
NUM_ITEMS = df['item_id'].max() + 1
EMBEDDING_DIM = 16
MAX_SEQ_LEN = 20

## 2. 数据集构建

我们需要构建两种格式的数据：
*   **DIN 数据**: `(User, Target_Item, History_Sequence) -> Label (Click)`
*   **MMOE 数据**: `(User, Item) -> [Label_Click, Label_Buy]`

In [ ]:
# 按时间排序
df['vtime'] = pd.to_datetime(df['vtime'])
df = df.sort_values(['user_id', 'vtime'])

# 构建用户历史行为序列
user_history = df.groupby('user_id')['item_id'].apply(list)

train_data = []
labels_ctr = [] # Click Through Rate
labels_cvr = [] # Conversion Rate (Buy)

# 假设 action 映射: 0: alipay(buy), 1: click, 2: collect, 3: cart (具体需根据实际编码调整，这里仅作演示假设)
# 实际 ID 编码后需要检查，这里简化处理：
# 我们假设 'click' 是样本，'alipay' 是正向转化。
# 真实场景中，通常取最后一次行为作为 Target，之前的作为 History。

for user, group in df.groupby('user_id'):
    items = group['item_id'].tolist()
    actions = group['action'].tolist()
    
    # 滑动窗口生成样本
    for i in range(1, len(items)):
        target_item = items[i]
        history_items = items[:i][-MAX_SEQ_LEN:] # 取最近 N 个
        
        # 模拟 Label: 真实场景需根据 action 判断
        # 这里为了演示模型跑通，随机生成 Label 或简单基于 action
        # 假设当前行是 click 则 ctr=1, 是 buy 则 cvr=1
        act = actions[i]
        is_click = 1 # 假设样本都是曝光
        is_buy = 1 if act == 0 else 0 # 假设 0 是购买 (仅作演示逻辑)
        
        train_data.append([user, target_item, history_items])
        labels_ctr.append(is_click)
        labels_cvr.append(is_buy)

# Padding
X_user = np.array([x[0] for x in train_data])
X_item = np.array([x[1] for x in train_data])
X_history = pad_sequences([x[2] for x in train_data], maxlen=MAX_SEQ_LEN, padding='post')

y_ctr = np.array(labels_ctr)
y_cvr = np.array(labels_cvr)

print(f"样本数量: {len(X_user)}")

## 3. 饿了么风格：Deep Interest Network (DIN)

核心创新点：**Attention Mechanism** (Local Activation Unit)。
模型会根据 `Target Item` 去加权 `History Items`。

In [ ]:
def build_din_model():
    # Inputs
    user_input = Input(shape=(1,), name='User_ID')
    item_input = Input(shape=(1,), name='Target_Item_ID')
    history_input = Input(shape=(MAX_SEQ_LEN,), name='History_Item_IDs')

    # Embedding Layer (Shared for Item and History)
    item_embedding_layer = layers.Embedding(NUM_ITEMS, EMBEDDING_DIM, name='Item_Embedding')
    
    # Lookups
    target_emb = item_embedding_layer(item_input) # (B, 1, E)
    history_emb = item_embedding_layer(history_input) # (B, T, E)
    
    # Attention Layer (Local Activation Unit)
    # Query: Target Item, Keys/Values: History Items
    
    # 简单 Attention 实现: Q * K -> Softmax -> Weights * V
    # DIN 原论文中使用了外积和拼接 [Q, K, Q-K, Q*K] 然后过 MLP
    
    # 这里实现简化版 Attention
    query = tf.tile(target_emb, [1, MAX_SEQ_LEN, 1]) # (B, T, E)
    concat = layers.Concatenate()([query, history_emb, query - history_emb, query * history_emb])
    att_weights = layers.Dense(32, activation='sigmoid')(concat)
    att_weights = layers.Dense(1)(att_weights) # (B, T, 1)
    
    # Masking (忽略 padding 的 0)
    # 在实际工程中需要处理 mask，这里简化略过
    
    # Weighted Sum
    user_interest = layers.Dot(axes=1)([att_weights, history_emb]) # (B, 1, E)
    user_interest = layers.Flatten()(user_interest)
    
    # Concat All Features
    target_vec = layers.Flatten()(target_emb)
    user_emb = layers.Flatten()(layers.Embedding(NUM_USERS, EMBEDDING_DIM)(user_input))
    
    concat_all = layers.Concatenate()([user_emb, user_interest, target_vec])
    
    # MLP
    x = layers.Dense(64, activation='relu')(concat_all)
    x = layers.Dense(32, activation='relu')(x)
    output = layers.Dense(1, activation='sigmoid', name='CTR_Output')(x)
    
    model = Model(inputs=[user_input, item_input, history_input], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

din_model = build_din_model()
# din_model.summary()

## 4. 美团风格：MMOE (Multi-gate Mixture-of-Experts)

核心创新点：**多任务学习**。用多个 Experts 提取特征，每个 Task (CTR, CVR) 有自己的 Gate 来组合这些 Experts。

In [ ]:
def build_mmoe_model(num_experts=3, num_tasks=2):
    # Inputs (简化为 User + Item)
    user_input = Input(shape=(1,), name='User_ID')
    item_input = Input(shape=(1,), name='Item_ID')
    
    # Embeddings & Concat
    u_emb = layers.Flatten()(layers.Embedding(NUM_USERS, EMBEDDING_DIM)(user_input))
    i_emb = layers.Flatten()(layers.Embedding(NUM_ITEMS, EMBEDDING_DIM)(item_input))
    input_layer = layers.Concatenate()([u_emb, i_emb])
    
    # Experts Layer
    expert_outputs = []
    for i in range(num_experts):
        expert = layers.Dense(32, activation='relu', name=f'Expert_{i}')(input_layer)
        expert_outputs.append(expert)
    # Stack experts: (B, Num_Experts, Hidden_Dim)
    experts_tensor = tf.stack(expert_outputs, axis=1) 
    
    # Multi-gate Layers
    final_outputs = []
    task_names = ['CTR', 'CVR']
    
    for i in range(num_tasks):
        # Gate for task i: 决定每个 Expert 的权重
        gate = layers.Dense(num_experts, activation='softmax', name=f'Gate_{task_names[i]}')(input_layer)
        gate = tf.expand_dims(gate, -1) # (B, Num_Experts, 1)
        
        # Weighted Sum of Experts
        task_input = tf.reduce_sum(experts_tensor * gate, axis=1) # (B, Hidden_Dim)
        
        # Tower for task i
        tower = layers.Dense(16, activation='relu')(task_input)
        output = layers.Dense(1, activation='sigmoid', name=f'Output_{task_names[i]}')(tower)
        final_outputs.append(output)
        
    model = Model(inputs=[user_input, item_input], outputs=final_outputs)
    # Loss weights 可以根据业务调整
    model.compile(optimizer='adam', 
                  loss={'Output_CTR': 'binary_crossentropy', 'Output_CVR': 'binary_crossentropy'}, 
                  metrics=['accuracy'])
    return model

mmoe_model = build_mmoe_model()

## 5. 模型训练演示

使用处理好的数据进行简单训练。

In [ ]:
print("Training DIN (Ele.me style)...")
din_model.fit(
    x=[X_user, X_item, X_history], 
    y=y_ctr, 
    batch_size=64, 
    epochs=1, 
    validation_split=0.1
)

print("\nTraining MMOE (Meituan style)...")
mmoe_model.fit(
    x=[X_user, X_item], 
    y=[y_ctr, y_cvr], 
    batch_size=64, 
    epochs=1, 
    validation_split=0.1
)